# 🏦 Predicción de Riesgo de Crédito con Inteligencia Artificial
## HE2 – Consultoría Económica con IA Responsable 2026-1

**Problema:** Las instituciones de microcrédito en Colombia pierden millones por mora crediticia debido a métodos de evaluación imprecisos.

**Solución:** Modelo de Machine Learning (Random Forest) que predice la probabilidad de incumplimiento crediticio con base en variables socioeconómicas y financieras.

**Dataset:** Give Me Some Credit – Kaggle (150,000 registros)

---


## 0. Librerías

In [4]:
import os
print(os.getcwd())

/Users/carolinarojas/Downloads


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print("✅ Librerías cargadas correctamente")

✅ Librerías cargadas correctamente


## 1. Carga y Exploración de Datos

Cargamos el dataset **Give Me Some Credit** de Kaggle.  
Contiene información financiera de 150,000 solicitantes de crédito con la variable objetivo `SeriousDlqin2yrs` (1 = entró en mora en los próximos 2 años).


In [2]:
df = pd.read_csv('cs-training.csv')
print("Filas y columnas:", df.shape)
print("\nPrimeras filas:")
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'cs-training.csv'

In [ ]:
print("=== TIPOS DE DATOS Y VALORES NULOS ===")
print(df.info())
print("\n=== ESTADÍSTICAS BÁSICAS ===")
df.describe()

## 2. Limpieza de Datos

Se identificaron valores faltantes en dos columnas:
- **MonthlyIncome**: 29,731 valores nulos (19.8%) → imputar con mediana
- **NumberOfDependents**: 3,924 valores nulos (2.6%) → imputar con mediana

Se usa la **mediana** (no la media) por ser más robusta ante valores atípicos extremos.


In [ ]:
# Verificar valores nulos
print("=== VALORES NULOS POR COLUMNA ===")
print(df.isnull().sum())

# Imputar con mediana
df['MonthlyIncome'] = df['MonthlyIncome'].fillna(df['MonthlyIncome'].median())
df['NumberOfDependents'] = df['NumberOfDependents'].fillna(df['NumberOfDependents'].median())

# Eliminar columna de índice innecesaria
df.drop(columns=['Unnamed: 0'], inplace=True)

print(f"\n✅ Datos limpios. Dimensiones: {df.shape}")
print(f"Valores nulos restantes: {df.isnull().sum().sum()}")

## 3. Análisis Exploratorio de Datos (EDA)

Visualizamos la distribución de las variables principales y su relación con la mora.

**Hallazgos clave:**
- Solo el **6.7%** de clientes entró en mora (dataset desbalanceado)
- Los **menores de 30 años** tienen la tasa de mora más alta (11.58%)
- La **edad** tiene correlación negativa con mora: a mayor edad, menor riesgo


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Análisis Exploratorio - Riesgo de Crédito', fontsize=16, fontweight='bold')

# 1. Distribución variable objetivo
axes[0,0].bar(['Sin mora (0)', 'Con mora (1)'],
               df['SeriousDlqin2yrs'].value_counts().sort_index(),
               color=['#2ecc71', '#e74c3c'])
axes[0,0].set_title('Distribución de Mora')
axes[0,0].set_ylabel('Cantidad de clientes')

# 2. Distribución de edad
axes[0,1].hist(df['age'], bins=30, color='#3498db', edgecolor='white')
axes[0,1].set_title('Distribución de Edad')
axes[0,1].set_xlabel('Edad')

# 3. Ingreso mensual
ingreso_filtrado = df[df['MonthlyIncome'] < 20000]['MonthlyIncome']
axes[0,2].hist(ingreso_filtrado, bins=30, color='#9b59b6', edgecolor='white')
axes[0,2].set_title('Ingreso Mensual (< $20,000)')
axes[0,2].set_xlabel('Ingreso')

# 4. Ratio de deuda
deuda_filtrada = df[df['DebtRatio'] < 2]['DebtRatio']
axes[1,0].hist(deuda_filtrada, bins=30, color='#e67e22', edgecolor='white')
axes[1,0].set_title('Ratio de Deuda')
axes[1,0].set_xlabel('Deuda / Ingreso')

# 5. Tasa de mora por grupo de edad
df['grupo_edad'] = pd.cut(df['age'], bins=[0,30,40,50,60,100],
                           labels=['<30','30-40','40-50','50-60','>60'])
mora_edad = df.groupby('grupo_edad')['SeriousDlqin2yrs'].mean() * 100
axes[1,1].bar(mora_edad.index, mora_edad.values, color='#e74c3c')
axes[1,1].set_title('Tasa de Mora por Grupo de Edad')
axes[1,1].set_ylabel('% en mora')

# 6. Correlaciones
corr = df[['SeriousDlqin2yrs','age','DebtRatio','MonthlyIncome',
           'NumberOfOpenCreditLinesAndLoans']].corr()
sns.heatmap(corr, annot=True, fmt='.2f', ax=axes[1,2], cmap='RdYlGn')
axes[1,2].set_title('Correlaciones')

plt.tight_layout()
plt.savefig('analisis_exploratorio.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Gráficas guardadas!")

## 4. Modelo de Inteligencia Artificial – Random Forest

### ¿Por qué Random Forest?
- Maneja bien datasets desbalanceados con `class_weight='balanced'`
- No requiere normalización de variables
- Proporciona importancia de variables interpretable
- Robusto ante valores atípicos

### División de datos
- **80% entrenamiento** (120,000 registros)
- **20% prueba** (30,000 registros)


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.preprocessing import LabelEncoder

# Codificar variable categórica
df['grupo_edad'] = LabelEncoder().fit_transform(df['grupo_edad'])

X = df.drop(columns=['SeriousDlqin2yrs'])
y = df['SeriousDlqin2yrs']

# División entrenamiento/prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Entrenamiento del modelo
modelo = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
modelo.fit(X_train, y_train)

# Evaluación
y_pred = modelo.predict(X_test)
print("=== RESULTADOS DEL MODELO ===")
print(classification_report(y_test, y_pred))
print("AUC-ROC:", round(roc_auc_score(y_test, modelo.predict_proba(X_test)[:,1]), 4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Resultados del Modelo - Random Forest', fontsize=14, fontweight='bold')

# Matriz de confusión
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', ax=axes[0], cmap='Blues',
            xticklabels=['Sin mora', 'Con mora'],
            yticklabels=['Sin mora', 'Con mora'])
axes[0].set_title('Matriz de Confusión')
axes[0].set_ylabel('Real')
axes[0].set_xlabel('Predicho')

# Importancia de variables
importancias = pd.Series(modelo.feature_importances_, index=X.columns)
importancias.sort_values().plot(kind='barh', ax=axes[1], color='#3498db')
axes[1].set_title('Importancia de Variables')
axes[1].set_xlabel('Importancia')

plt.tight_layout()
plt.savefig('resultados_modelo.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Gráficas del modelo guardadas!")

## 5. Análisis Económico – VPN y TIR

Cuantificamos el valor económico de implementar el modelo en una institución de microcrédito colombiana con $10,000 millones COP en cartera.

**Lógica del ahorro:** Si el modelo reduce la mora un 25%, el ahorro anual es la diferencia entre la mora evitada multiplicada por la pérdida promedio por crédito en mora.


In [ ]:
import numpy as np
from numpy_financial import irr

# Supuestos
cartera = 10_000_000_000
tasa_mora_actual = 0.067
tasa_mora_modelo = 0.067 * 0.75   # reducción del 25%
perdida_por_mora = 0.60
costo_implementacion = 150_000_000
costo_mantenimiento = 40_000_000
tasa_descuento = 0.12

# Ahorro anual
ahorro_anual = (tasa_mora_actual - tasa_mora_modelo) * cartera * perdida_por_mora

# Flujos de caja (5 años)
flujos = [-costo_implementacion]
for i in range(1, 6):
    costo = costo_mantenimiento if i > 1 else 0
    flujos.append(ahorro_anual - costo)

# VPN y TIR
vpn = sum(f / (1 + tasa_descuento)**i for i, f in enumerate(flujos))
tir = irr(flujos)

print("=== ANÁLISIS ECONÓMICO ===")
print(f"Tasa mora actual:        {tasa_mora_actual*100:.1f}%")
print(f"Tasa mora con modelo:    {tasa_mora_modelo*100:.2f}%")
print(f"Ahorro anual estimado:   ${ahorro_anual/1e6:.1f} millones COP")
print(f"\nFlujos de caja (5 años): {[round(f/1e6,1) for f in flujos]} M COP")
print(f"\nVPN (12%):  ${vpn/1e6:.1f} millones COP")
print(f"TIR:        {tir*100:.1f}%")
print(f"\n{'✅ PROYECTO VIABLE' if vpn > 0 else '❌ PROYECTO NO VIABLE'}")

### 5.1 Análisis de Sensibilidad

Evaluamos tres escenarios según la efectividad del modelo en reducir la mora:

In [ ]:
escenarios = {
    'Pesimista (10% reducción mora)': 0.10,
    'Base (25% reducción mora)':      0.25,
    'Optimista (40% reducción mora)': 0.40
}

print("=== ANÁLISIS DE SENSIBILIDAD ===\n")
print(f"{'Escenario':<35} {'Ahorro Anual':>15} {'VPN':>15} {'TIR':>8}")
print("-" * 75)

for nombre, reduccion in escenarios.items():
    ahorro = tasa_mora_actual * reduccion * cartera * perdida_por_mora
    fl = [-costo_implementacion] + [ahorro - (costo_mantenimiento if i > 0 else 0) for i in range(5)]
    v = sum(f / (1 + tasa_descuento)**i for i, f in enumerate(fl))
    t = irr(fl)
    print(f"{nombre:<35} ${ahorro/1e6:>13.1f}M ${v/1e6:>13.1f}M {t*100:>7.1f}%")

print("\n✅ Análisis de sensibilidad completo!")

## 6. Análisis de Sesgo y IA Responsable

Evaluamos si el modelo trata de forma equitativa a diferentes grupos etarios.

**Métrica:** Diferencia entre tasa de mora real y predicha por grupo de edad.
- Valor negativo = modelo subestima mora (más permisivo de lo que debería)
- Valor positivo = modelo sobreestima mora (más restrictivo de lo que debería)


In [ ]:
X_test_df = X_test.copy()
X_test_df['real'] = y_test.values
X_test_df['predicho'] = y_pred
X_test_df['grupo_edad'] = pd.cut(X_test_df['age'],
                                  bins=[0,30,40,50,60,100],
                                  labels=['<30','30-40','40-50','50-60','>60'])

sesgo = X_test_df.groupby('grupo_edad').apply(
    lambda g: pd.Series({
        'tasa_mora_real': g['real'].mean()*100,
        'tasa_mora_predicha': g['predicho'].mean()*100,
        'diferencia': (g['predicho'].mean() - g['real'].mean())*100
    })
).round(2)

print("=== ANÁLISIS DE SESGO POR EDAD ===")
print(sesgo)
print("\n⚠️  El modelo subestima mora en todos los grupos.")
print("Los jóvenes (<30) son el grupo más afectado (-8.06 pp).")
print("Se recomienda supervisión humana obligatoria en este segmento.")

## 7. Demo en Vivo – Predicción de Cliente Nuevo

Esta sección simula el uso real del modelo: dado un perfil de cliente, el modelo calcula la probabilidad de mora y emite una recomendación de aprobación o rechazo.


In [ ]:
import joblib

# Guardar modelo entrenado
joblib.dump(modelo, 'modelo_riesgo_credito.pkl')
print("✅ Modelo guardado como 'modelo_riesgo_credito.pkl'\n")

# Simular predicción de cliente nuevo
cliente_nuevo = pd.DataFrame({
    'RevolvingUtilizationOfUnsecuredLines': [0.85],
    'age': [28],
    'NumberOfTime30-59DaysPastDueNotWorse': [2],
    'DebtRatio': [0.6],
    'MonthlyIncome': [2500000],
    'NumberOfOpenCreditLinesAndLoans': [4],
    'NumberOfTimes90DaysLate': [0],
    'NumberRealEstateLoansOrLines': [0],
    'NumberOfTime60-89DaysPastDueNotWorse': [1],
    'NumberOfDependents': [1],
    'grupo_edad': [0]
})

prob_mora = modelo.predict_proba(cliente_nuevo)[0][1]
decision = "❌ RECHAZAR – Requiere revisión manual" if prob_mora > 0.15 else "✅ APROBAR"

print("=== PERFIL DEL CLIENTE ===")
print(f"Edad: 28 años | Ingreso: $2,500,000 COP | Ratio deuda: 0.6")
print(f"Atrasos previos: 2 veces (30-59 días) | 1 vez (60-89 días)")
print(f"\n=== RESULTADO DEL MODELO ===")
print(f"Probabilidad de mora: {prob_mora*100:.1f}%")
print(f"Decisión:             {decision}")

---
## Resumen de Resultados

| Métrica | Valor |
|---------|-------|
| Dataset | 150,000 registros |
| AUC-ROC | 0.83 |
| Accuracy | 94% |
| Ahorro anual estimado | $100.5M COP |
| VPN (escenario base) | +$103.8M COP |
| TIR | 40.8% |
| Sesgo más alto | Jóvenes <30 años (-8.06 pp) |

**Conclusión:** El modelo es técnicamente sólido y económicamente viable bajo el escenario base. Se recomienda implementación piloto con supervisión humana obligatoria, especialmente para solicitantes jóvenes.
